# 한국어 녹음 파일 자동 요약

**3단계 파이프라인:**
1. 각 m4a 파일 → Speech-to-Text (OpenAI Whisper)
2. 각 텍스트 개별 정리 (Claude)
3. 전체 통합 요약 (Claude)

In [1]:
# ── 최초 1회만 실행 ────────────────────────────────────────────────────────
import os
from pathlib import Path

# 패키지 설치
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "anthropic", "openai-whisper"], check=True)

# ffmpeg 경로 등록 (노트북과 같은 폴더에 있는 ffmpeg 사용)
_ffmpeg_bin = Path("ffmpeg-8.1.1-essentials_build/bin").resolve()
os.environ["PATH"] = str(_ffmpeg_bin) + os.pathsep + os.environ["PATH"]

print("초기 설정 완료")

초기 설정 완료


In [4]:
import os
import whisper
from pathlib import Path

# Records 루트 디렉토리 (서브폴더들이 들어있는 폴더)
AUDIO_DIR = "./Records"

## Step 1: 음성 → 텍스트 변환 (Whisper STT)

In [7]:
import ssl
ssl._create_default_https_context = ssl._create_unverified_context

# 모델 크기: tiny < base < small < medium < large (클수록 정확하지만 느림)
print("Whisper 모델 로드 중 ...")
whisper_model = whisper.load_model("medium")
print("완료")

Whisper 모델 로드 중 ...


URLError: <urlopen error [SSL: WRONG_VERSION_NUMBER] wrong version number (_ssl.c:1006)>

In [ ]:
def transcribe_audio(audio_path: str) -> str:
    """m4a 파일을 한국어 텍스트로 변환 (로컬 Whisper)"""
    result = whisper_model.transcribe(audio_path, language="ko")
    return result["text"]


# 서브폴더 목록 동적 탐색
subfolders = sorted(
    [d for d in Path(AUDIO_DIR).iterdir() if d.is_dir()],
    key=lambda d: d.name,
)
if not subfolders:
    raise FileNotFoundError(f"'{AUDIO_DIR}' 아래에 서브폴더가 없습니다.")

# 각 서브폴더의 m4a 파일 목록
folder_files: dict[str, list[str]] = {}
for sub in subfolders:
    files = sorted(sub.glob("*.m4a"))
    if files:
        folder_files[sub.name] = [str(p) for p in files]

print(f"폴더 {len(folder_files)}개 발견:")
for fname, files in folder_files.items():
    print(f"  [{fname}]  {len(files)}개 파일")
print()

# 모든 파일 STT 변환
transcriptions: dict[str, dict[str, str]] = {}
for folder_name, paths in folder_files.items():
    transcriptions[folder_name] = {}
    for path in paths:
        name = Path(path).name
        print(f"변환 중: [{folder_name}] {name} ...", end=" ", flush=True)
        text = transcribe_audio(path)
        transcriptions[folder_name][name] = text
        print(f"완료 ({len(text):,}자)")
    print()

print("[Step 1 완료] 전체 음성 → 텍스트 변환 완료")

## Step 2: 파일별 개별 정리 (Claude)

In [ ]:
ANTHROPIC_API_KEY = os.environ.get("ANTHROPIC_API_KEY", "your-anthropic-api-key")
anthropic_client = anthropic.Anthropic(api_key=ANTHROPIC_API_KEY)

In [ ]:
def summarize_file(filename: str, text: str) -> str:
    """개별 파일 텍스트를 핵심 위주로 정리"""
    prompt = f"""다음은 '{filename}' 녹음 파일의 텍스트입니다.
내용을 한국어로 명확하게 정리해 주세요.

요구사항:
- 핵심 주제와 논점 파악
- 중요 내용을 논리적 순서로 정리
- 불필요한 반복·잡음 제거, 원문 의미 유지
- 읽기 쉬운 단락 구조로 작성

텍스트:
{text}"""

    with anthropic_client.messages.stream(
        model="claude-opus-4-8",
        max_tokens=4096,
        thinking={"type": "adaptive"},
        messages=[{"role": "user", "content": prompt}],
    ) as stream:
        return stream.get_final_message().content[-1].text


# summaries: {폴더명: {파일명: 요약}}
summaries: dict[str, dict[str, str]] = {}
for folder_name, files_dict in transcriptions.items():
    summaries[folder_name] = {}
    for filename, text in files_dict.items():
        print(f"정리 중: [{folder_name}] {filename} ...", end=" ", flush=True)
        summary = summarize_file(filename, text)
        summaries[folder_name][filename] = summary
        print("완료")
    print()

print("[Step 2 완료] 파일별 개별 정리 완료")

## Step 3: 전체 흐름 연결 및 최종 통합 요약 (Claude)

In [ ]:
def create_folder_summary(folder_name: str, files_dict: dict[str, str]) -> str:
    """한 폴더(주차) 내 파일들의 정리본을 통합하여 최종 요약 생성"""
    combined = "\n\n".join(
        f"=== {fname} ===\n{text}" for fname, text in files_dict.items()
    )

    prompt = f"""다음은 '{folder_name}'의 여러 녹음 파일에서 정리된 내용입니다.
이 내용들을 통합하여 전체적인 흐름을 연결하는 최종 정리본을 한국어로 작성해 주세요.

요구사항:
- 파일 간 내용의 연결 고리와 발전 흐름 파악
- 반복되는 핵심 주제는 통합하여 하나로 정리
- 전체 논의의 맥락과 결론을 명확히 제시
- 논리적이고 읽기 쉬운 구조로 통합

개별 파일 정리 내용:
{combined}"""

    with anthropic_client.messages.stream(
        model="claude-opus-4-8",
        max_tokens=8192,
        thinking={"type": "adaptive"},
        messages=[{"role": "user", "content": prompt}],
    ) as stream:
        return stream.get_final_message().content[-1].text


# folder_summaries: {폴더명: 통합 요약}
folder_summaries: dict[str, str] = {}
for folder_name, files_dict in summaries.items():
    print(f"통합 요약 중: [{folder_name}] ...", end=" ", flush=True)
    folder_summaries[folder_name] = create_folder_summary(folder_name, files_dict)
    print("완료")

print("\n[Step 3 완료] 폴더별 통합 요약 완료")

for folder_name, summary in folder_summaries.items():
    print(f"\n{'='*60}")
    print(f"[{folder_name}] 최종 요약")
    print("=" * 60)
    print(summary)

## 결과 저장

In [ ]:
OUTPUT_DIR = "./output"

for folder_name in transcriptions:
    folder_out = os.path.join(OUTPUT_DIR, folder_name)
    os.makedirs(folder_out, exist_ok=True)

    # 원본 텍스트 저장
    for name, text in transcriptions[folder_name].items():
        out = os.path.join(folder_out, f"{Path(name).stem}_transcription.txt")
        with open(out, "w", encoding="utf-8") as f:
            f.write(text)

    # 개별 정리본 저장
    for name, summary in summaries[folder_name].items():
        out = os.path.join(folder_out, f"{Path(name).stem}_summary.txt")
        with open(out, "w", encoding="utf-8") as f:
            f.write(summary)

    # 폴더별 최종 요약 저장
    out = os.path.join(folder_out, "final_summary.txt")
    with open(out, "w", encoding="utf-8") as f:
        f.write(folder_summaries[folder_name])

    file_count = len(transcriptions[folder_name])
    print(f"[{folder_name}] 저장 완료 ({file_count}개 파일)")

print(f"\n전체 저장 완료 → '{OUTPUT_DIR}/' 디렉토리")
print("구조: output/<폴더명>/*_transcription.txt, *_summary.txt, final_summary.txt")